In [1]:
"""
Baseline Model Training & Evaluating

This notebook established the baseline performance for our prediction task. Before applying advanced feature selection or complex algorithms like XGBoost, we train a standard Random Forest Classifier on all available numeric features. This gives us a benchmark to measure future improvements against.

Key steps:
+ Isolating numeric features and handling missing values (basic imputation);
+ Creating train and test sets using stratified sampling to preserve the extreme class imbalance of Oscar winners;
+ Training a baseline Random Forest model;
+ Generating a classification report with a focus on minority class metrics (Recall and F1-score).
"""

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import time

In [2]:
# Load the cleaned and processed dataset from the previous notebook
df = pd.read_csv('../data/processed/oscar_tmdb_cleaned.csv')

In [3]:
"""
Tree-based model require strictly numerical inputs. We isolate numeric columns, effectively dropping any remaining raw text fields.
"""

numeric_df = df.select_dtypes(include=['number'])

In [4]:
# Define features (X) and the target variable (y)
x = numeric_df.drop(columns=['is_oscar_winner', 'id'], errors='ignore')
y = numeric_df['is_oscar_winner']

In [5]:
x = x.fillna(0)  # Basic missing value imputation

In [6]:
x.shape[0]

985

In [7]:
x.shape[1]  # Total number of features

4710

In [8]:
"""
We use a 80/20 split.
CRITICAL: 'stratify=y' ensures that the train and test sets have the exact same proportion of Oscar winners (class 1) as the original dataset. This is vital for highly imbalanced datasets.
"""

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
start_time = time.time()

# n_jobs=-1 utilizes all available CPU cores to speed up training
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(x_train, y_train)

train_time = time.time() - start_time

train_time

0.20487260818481445

In [10]:
y_pred = rf_model.predict(x_test)

In [11]:
"""
BASELINE model results (BEFORE feature selection or hyperparameter tuning). We convert the classification report into a Pandas DataFrame for cleaner visualization in the Jupyter Notebook environment.
"""

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")

report_dict = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report_dict).T

report_df.round(3)

Accuracy: 0.6497

Classification Report:


,precision,recall,f1-score,support
0,0.644,0.950,0.768,120.00
1,0.700,0.182,0.289,77.00
accuracy,0.650,0.650,0.650,0.65
macro avg,0.672,0.566,0.528,197.00
weighted avg,0.666,0.650,0.580,197.00
